# AGEMR B3 v3：各年龄样本 GWAS → P 值选工具 → two-sample MR

固定 BP target，在不同年龄构成的 BMI source 中分别做 pooled GWAS、按各自的 P 值选工具，再跑 BMI→SBP/DBP 的 IVW、dIVW、weighted median、MR-Egger。

**不设独立 discovery 组，不重新 pruning/clumping。** 按用户要求沿用现有已做 LD pruning 的候选面板。每个 source 的工具可以不同；比较反映年龄构成、SNP–BMI 系数和工具集合共同变化。它仍是候选面板分析，不是全基因组扫描。

默认工具规则：有效 pooled BMI GWAS ADD 行、`P < 1e-5`。不额外按 F、MAF、MAC 筛选，不按 BP P 值选工具，不因工具少自动放宽 P。F/MAC 作为诊断保留。同一 source 用于 GWAS 筛选和估计 βX，winner's curse 未被消除，属于本版复现的常规流程。

人员按固定 IID hash 分为 target（20%）和 source（80%）。沿用 v2 的 target hash 区间和抽样随机数：输入、seed 和上限相同时，BP target 与 v2 相同。原 discovery 角色全部并回 source；未抽进最终 target 的 target 角色也不会进入 source，保证两样本不重叠。

五种 source 年龄方案不变，每组默认最多 30,000 人，3 次配对重复，共同年龄支持不足时统一降低人数。MR 没有使用 IPW。协变量仍是原保存的 age spline、sex、QPC 完整空间。

在 Workbench 核对配置后 Run All。缺 PLINK2 时沿用旧版的公开软件恢复逻辑；只读已有 cohort/PGEN。新输出位于 `B3_MR_source_selected/`，不覆盖旧结果。真实 AoU 数据需在 Workbench 运行，本版交付不附真实结果。

In [ ]:
from pathlib import Path
import json, hashlib, uuid, subprocess, shutil, warnings, gc, tempfile, os
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy import stats, linalg, sparse
import matplotlib.pyplot as plt
from IPython.display import display

# Only this configuration cell normally needs editing.
LINEAR_RUN = None
B1_ROOT = Path.home()/"workspace/agemr-data/final_cohort_v1/B1_age_varying"
PLINK_EXECUTABLE = None  # e.g. str(Path.home()/"plink2_env/bin/plink2")
INSTALL_PLINK_IF_MISSING = True
PLINK_ENV = Path(tempfile.gettempdir())/("agemr-tools-"+str(os.getuid()))/"plink2_2.0.0a.7.8"
UNRELATED_KEEP = None  # Optional existing IID or FID=0 keep; no new kinship inference.
SEED = 20261028
N_TARGET_REQUESTED = 30000
N_SOURCE_REQUESTED = 30000
N_REPEATS = 3
N_BOOT = 2000
IV_P_THRESHOLD = 1e-5  # Sole relevance filter: current source ordinary GWAS P, strict <.
MAX_UNION_IV = 128  # Covariance memory guard only; do not truncate selected SNPs.
MIN_ROLE_N = 100
THREADS = 2
MEMORY_MB = 1024
AGE_BIN_YEARS = 5
SCENARIOS = [("matched", 0.), ("younger_moderate", -.5),
             ("younger_strong", -1.), ("older_moderate", .5), ("older_strong", 1.)]
OUTCOMES = ("SBP", "DBP")
METHODS = ("IVW", "dIVW", "weighted_median", "MR_Egger")
KEY = ["#CHROM", "POS", "ID", "REF", "ALT"]
CONTRAST_FAMILY = len(OUTCOMES)*(len(SCENARIOS)-1)*len(METHODS)

def require(ok, message):
    if not ok: raise RuntimeError(message)

def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(1024*1024), b""): h.update(b)
    return h.hexdigest()

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, default=str))

def iid_table(path):
    x = pd.read_csv(path, sep=r"\s+", dtype=str, keep_default_na=False)
    x = x.rename(columns={"#IID":"IID", "#FID":"FID"})
    require("IID" in x and x.IID.is_unique, f"Missing/duplicate IID: {path}")
    if "FID" in x:
        require(x.FID.eq("0").all(), "This workflow requires IID-only or FID=0 inputs")
        x = x.drop(columns="FID")
    return x.set_index("IID")

def stable_rng(label):
    value = int.from_bytes(hashlib.sha256(f"{SEED}|{label}".encode()).digest()[:8], "big")
    return np.random.default_rng(value)

def role(iid):
    u = int.from_bytes(hashlib.sha256(f"B3|{SEED}|{iid}".encode()).digest()[:8], "big")/2**64
    return "target" if .35 <= u < .55 else "source"

def run_cmd(command, prefix):
    prefix = Path(prefix); prefix.parent.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    write_json(str(prefix)+".command.json", command)
    with Path(str(prefix)+".stdout.txt").open("w") as out, Path(str(prefix)+".stderr.txt").open("w") as err:
        res = subprocess.run(command, stdout=out, stderr=err)
    if res.returncode:
        detail = Path(str(prefix)+".stderr.txt").read_text(errors="replace")[-2500:]
        raise RuntimeError(f"PLINK failed ({res.returncode}): {prefix}\n{detail}")

def nuisance_basis(C):
    X = np.column_stack([np.ones(len(C)), np.asarray(C, float)])
    Q, R, piv = linalg.qr(X, mode="economic", pivoting=True, check_finite=False)
    diag = np.abs(np.diag(R))
    rank = int((diag > max(X.shape)*np.finfo(float).eps*max(diag.max(), 1)).sum())
    return Q[:, :rank], rank, piv[:rank]

require(N_BOOT >= 200 and N_REPEATS >= 1, "Use at least 200 draws and one repeat")
if LINEAR_RUN is None:
    choices = []
    for marker in (B1_ROOT/"B1_linear_age").glob("*/RUN_COMPLETE.json"):
        m = json.loads(marker.read_text())
        if m.get("status") == "complete" and all((marker.parent/"inputs"/f"{t}.meta.json").is_file() for t in ("BMI",)+OUTCOMES):
            choices.append((m.get("finished_utc", ""), marker.parent))
    require(bool(choices), "No completed B1 linear run; set LINEAR_RUN to the existing run")
    LINEAR_RUN = max(choices, key=lambda z:z[0])[1]
else: LINEAR_RUN = Path(LINEAR_RUN).expanduser()
completion = json.loads((LINEAR_RUN/"RUN_COMPLETE.json").read_text())
require(completion.get("status") == "complete", "Selected run is incomplete")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:6]
OUT = LINEAR_RUN/"B3_MR_source_selected"/stamp
for name in ("qc", "design", "source_GWAS", "genotypes", "associations", "MR", "figures"):
    (OUT/name).mkdir(parents=True, exist_ok=True)
print("B3 v3 source-selected MR - existing results:", LINEAR_RUN, flush=True)
print("New B3 outputs and software diagnostics:", OUT, flush=True)

# Embedded in each delivered notebook; no external Python file is needed.
def resolve_existing_or_install_plink(preferred, linear_run, log_dir, install_if_missing, env_prefix):
    import os, re, sys, tempfile
    log_dir = Path(log_dir); log_dir.mkdir(parents=True, exist_ok=True)
    env_prefix = Path(env_prefix).expanduser()
    audit, candidates, seen = [], [], set()

    def add(value, origin):
        if value is not None and str(value).strip():
            value = os.path.expandvars(str(value))
            found = shutil.which(value) if not Path(value).is_absolute() else None
            path = str(Path(found or value).expanduser().absolute())
            if path not in seen:
                seen.add(path); candidates.append((path, origin))

    def save():
        (log_dir/"plink_discovery.json").write_text(json.dumps(audit, indent=2))

    add(preferred, "explicit configuration")
    # Read program paths only; never replay a historical command or use shell=True.
    run = Path(linear_run)
    old_audit = run/"qc/plink_discovery.json"
    if old_audit.is_file():
        try:
            old = json.loads(old_audit.read_text())
            if isinstance(old, list):
                for r in old:
                    if isinstance(r, dict) and r.get("status") == "usable":
                        add(r.get("path"), "B1 successful discovery record")
        except (OSError, ValueError): pass
    historical = list(run.glob("*_results/*.command.json"))
    historical += list((run/"main_effect_stratified_proportions/pooled_main").glob("*/*.successful_signature.json"))
    for path in sorted(historical)[:200]:
        try:
            entry = json.loads(path.read_text())
            command = entry.get("command") if isinstance(entry, dict) else entry
            if isinstance(command,list) and command and isinstance(command[0],str):
                if Path(command[0]).name.lower() in ("plink2", "plink2.exe"):
                    add(command[0], "previous successful/run command: "+str(path))
        except (OSError, ValueError): continue
    for p in [shutil.which("plink2"), env_prefix/"bin/plink2",
              Path.home()/"plink2_env/bin/plink2", Path(sys.prefix)/"bin/plink2",
              Path.home()/".local/bin/plink2", "/home/dataproc/plink2_env/bin/plink2",
              "/home/jupyter/plink2_env/bin/plink2", "/opt/conda/default/bin/plink2",
              "/opt/conda/bin/plink2", "/usr/local/bin/plink2", "/usr/bin/plink2"]:
        add(p, "known software location")
    if env_prefix.parent.is_dir():
        for p in sorted(env_prefix.parent.glob("plink2_*/bin/plink2"))[:50]:
            add(p, "cached driver-local software")

    def probe(path, origin):
        r = {"path":str(path), "origin":origin}
        try:
            if not Path(path).is_file(): r["status"] = "missing"
            else:
                r["executable_mode"] = os.access(path,os.X_OK)
                v = subprocess.run([str(path),"--version"],capture_output=True,text=True,timeout=30)
                r["returncode"] = v.returncode
                r["version_or_error"] = (v.stdout+v.stderr).strip()[-2500:]
                r["status"] = "usable" if v.returncode == 0 and re.search(r"PLINK\s+v?2\.",r["version_or_error"],re.I) else "startup_failed"
        except (OSError, subprocess.TimeoutExpired) as e:
            r.update(status="execution_failed", error=str(e), errno=getattr(e,"errno",None))
        audit.append(r); save()
        if r["status"] == "usable": return str(path)
        return None

    def finish(path):
        print("Using PLINK2:",path,flush=True)
        print(audit[-1].get("version_or_error",""),flush=True)
        (log_dir/"plink_selected.json").write_text(json.dumps(audit[-1],indent=2))
        return path

    for path,origin in candidates:
        found = probe(path,origin)
        if found: return finish(found)
    important = [r for r in audit if r["status"] != "missing"]
    print("PLINK2 search diagnostics:",flush=True)
    print(pd.DataFrame(important or audit).to_string(index=False),flush=True)
    if not install_if_missing:
        raise RuntimeError(f"No runnable PLINK2; diagnostics: {log_dir/'plink_discovery.json'}. Set PLINK_EXECUTABLE or INSTALL_PLINK_IF_MISSING=True.")

    manager = None
    for p in [shutil.which("mamba"), shutil.which("conda"),
              "/opt/conda/default/bin/mamba", "/opt/conda/default/bin/conda", "/opt/conda/bin/conda"]:
        if p and Path(p).is_file() and os.access(p,os.X_OK): manager = str(p); break
    if manager is None:
        raise RuntimeError(f"No runnable PLINK2 and no mamba/conda available. Inspect {log_dir/'plink_discovery.json'}; set PLINK_EXECUTABLE to the working binary.")
    # Do not overwrite a broken prefix or mutate the Spark/Python environment.
    destination = env_prefix
    if destination.exists(): destination = destination.with_name(destination.name+"_"+uuid.uuid4().hex[:8])
    destination.parent.mkdir(parents=True,exist_ok=True)
    command = [manager,"create","-y","-p",str(destination),"--override-channels",
               "-c","conda-forge","-c","bioconda","plink2=2.0.0a.7.8"]
    (log_dir/"plink_install.command.json").write_text(json.dumps(command,indent=2))
    print("Installing public PLINK2 software only into driver-local:",destination,flush=True)
    print("Genotype/phenotype files remain in the workspace; active Python/Spark environment is unchanged.",flush=True)
    with (log_dir/"plink_install.log").open("w") as log:
        proc = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in proc.stdout:
                print(line,end="",flush=True); log.write(line); log.flush()
            returncode = proc.wait()
        except BaseException:
            proc.terminate()
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired: proc.kill(); proc.wait()
            raise
    if returncode:
        raise RuntimeError(f"Public software installation failed ({returncode}). See {log_dir/'plink_install.log'}; no genotype analysis has started.")
    found = probe(destination/"bin/plink2","new public software installation")
    if not found:
        raise RuntimeError("PLINK2 was installed but cannot start: "+json.dumps(audit[-1])+f". See {log_dir/'plink_discovery.json'}")
    return finish(found)

PLINK = resolve_existing_or_install_plink(PLINK_EXECUTABLE,LINEAR_RUN,OUT/"qc",INSTALL_PLINK_IF_MISSING,PLINK_ENV)
PLINK_VERSION = json.loads((OUT/"qc/plink_selected.json").read_text())["version_or_error"]

records = json.loads((LINEAR_RUN/"qc/input_files.json").read_text())
pfiles = {int(Path(r["path"]).stem.replace("BMI_chr", "")):Path(r["path"]).with_suffix("")
          for r in records if Path(r["path"]).suffix == ".pgen"}
for record in records:
    p = Path(record["path"])
    if p.suffix in (".pgen", ".pvar", ".psam"):
        st = p.stat()
        require(st.st_size == record["bytes"] and st.st_mtime_ns == record["mtime_ns"], f"Genotype input changed: {p}")
panel = pd.read_csv(LINEAR_RUN/"qc/variant_panel.tsv", sep="\t", dtype={"#CHROM":str})
panel["#CHROM"] = panel["#CHROM"].str.replace(r"^chr", "", regex=True)
require(panel.ID.is_unique and not panel.duplicated(KEY).any(), "Ambiguous panel identifiers")
require(panel["#CHROM"].astype(int).between(1,22).all() and not panel.ALT.str.contains(",").any(), "Autosomal biallelic panel required")

pheno, covar, age, input_audit = {}, {}, {}, []
unrelated = None if UNRELATED_KEEP is None else set(iid_table(Path(UNRELATED_KEEP)).index)
for trait in ("BMI",)+OUTCOMES:
    pp = LINEAR_RUN/"inputs"/f"{trait}.pheno.tsv"
    cp = LINEAR_RUN/"inputs"/f"{trait}.covar.tsv"
    mp = LINEAR_RUN/"inputs"/f"{trait}.meta.json"
    meta = json.loads(mp.read_text())
    require(sha(pp) == meta["pheno_sha256"] and sha(cp) == meta["covar_sha256"], f"{trait}: saved input changed")
    ph = iid_table(pp)[[trait]].apply(pd.to_numeric, errors="raise").astype(float)
    cv = iid_table(cp).loc[ph.index].apply(pd.to_numeric, errors="raise").astype(float)
    if unrelated is not None: ph = ph.loc[ph.index.isin(unrelated)]; cv = cv.loc[ph.index]
    require(np.isfinite(ph.to_numpy()).all() and np.isfinite(cv.to_numpy()).all(), f"Missing {trait} input")
    require("AGE10" in cv and meta["age_unit_years"] == 10, "Need saved AGE10 and age center metadata")
    pheno[trait], covar[trait] = ph[trait], cv
    age[trait] = cv.AGE10*float(meta["age_unit_years"])+float(meta["age_center_years"])
    input_audit.append({"trait":trait, "N":len(ph), "pheno_sha256":sha(pp), "covar_sha256":sha(cp), "meta_sha256":sha(mp)})
pd.DataFrame(input_audit).to_csv(OUT/"qc/input_hashes.tsv", sep="\t", index=False)
require(0 < IV_P_THRESHOLD < 1, "P threshold must be between zero and one")

## 1. 固定 BP target，抽取五种年龄构成

每 5 岁一个年龄箱，以 target 比例乘指数倾斜因子再归一化：κ=0、±0.5、±1。同一次重复共享箱内随机排序，source 之间可以重叠；target 与 source 不重叠。每组人数相同，保存并显示各年龄箱的实际比例，不只显示平均年龄。年龄均为相应 BMI/BP 的测量年龄。

ESS 只作支持度诊断，未用于加权 GWAS/MR。

In [ ]:
all_ids = pd.Index(sorted(set().union(*(set(y.index) for y in pheno.values()))), name="IID")
roles = pd.Series([role(i) for i in all_ids], index=all_ids, name="role")
roles.to_csv(OUT/"design/individual_roles.tsv", sep="\t", index_label="#IID")
bp_ids = pheno["SBP"].index.intersection(pheno["DBP"].index).sort_values()
require(np.allclose(age["SBP"].loc[bp_ids], age["DBP"].loc[bp_ids], atol=1e-7, rtol=0), "SBP/DBP ages differ; use separate target designs")
target_pool = bp_ids[roles.loc[bp_ids].eq("target")]
source_pool = pheno["BMI"].index[roles.loc[pheno["BMI"].index].eq("source")].sort_values()
require(min(len(target_pool), len(source_pool)) >= MIN_ROLE_N, "At least one role pool too small")
target_ids = pd.Index(stable_rng("target").choice(target_pool, min(N_TARGET_REQUESTED, len(target_pool)), replace=False)).sort_values()
require(not set(target_ids)&set(source_pool), "Role overlap")
for name, ids in [("target", target_ids), ("source_pool", source_pool)]:
    pd.DataFrame({"#IID":ids}).to_csv(OUT/"design"/f"{name}.keep.tsv", sep="\t", index=False)

target_age = age["SBP"].loc[target_ids].to_numpy()
source_age = age["BMI"].loc[source_pool].to_numpy()
require(np.std(target_age, ddof=1) > 0, "No target age variation")
age_min = np.floor(min(target_age.min(), source_age.min())/AGE_BIN_YEARS)*AGE_BIN_YEARS
age_max = np.floor(max(target_age.max(), source_age.max())/AGE_BIN_YEARS)*AGE_BIN_YEARS+AGE_BIN_YEARS
edges = np.arange(age_min, age_max+AGE_BIN_YEARS*.5, AGE_BIN_YEARS)
target_bin = np.searchsorted(edges, target_age, side="right")-1
source_bin = np.searchsorted(edges, source_age, side="right")-1
nbins = len(edges)-1
target_count = np.bincount(target_bin, minlength=nbins)
pool_count = np.bincount(source_bin, minlength=nbins)
pi = target_count/len(target_ids)
active = pi > 0
require((pool_count[active] > 0).all(), "Target age support missing in source pool; inspect design before changing bins")
centers = (edges[:-1]+edges[1:])/2
zcenters = (centers-target_age.mean())/target_age.std(ddof=1)
probabilities = {}
for name, tilt in SCENARIOS:
    q = pi*np.exp(tilt*zcenters)
    probabilities[name] = q/q.sum()
capacity = min(float(np.min(pool_count[active]/q[active])) for q in probabilities.values())
N_SOURCE = min(N_SOURCE_REQUESTED, int(.9*capacity))
require(N_SOURCE >= MIN_ROLE_N, "Insufficient shared age support for equal-sized sources")
if N_SOURCE < N_SOURCE_REQUESTED:
    warnings.warn(f"Common source N reduced once to {N_SOURCE} for age-bin capacity; fixed across all scenarios/repeats")

def quotas(q, n):
    v = n*np.asarray(q); a = np.floor(v).astype(int)
    extra = n-a.sum()
    if extra: a[np.argsort(-(v-a), kind="stable")[:extra]] += 1
    require(a.sum() == n and (a <= pool_count).all(), "Age-bin quota exceeds source capacity")
    return a

quota_rows, sample_rows, source_sets = [], [], []
for rep in range(N_REPEATS):
    # Common within-bin rankings make matched/mismatched comparisons paired.
    rankings = {b:stable_rng(f"source|{rep}|{b}").permutation(source_pool[source_bin == b]) for b in range(nbins)}
    for name, tilt in SCENARIOS:
        q = probabilities[name]; a = quotas(q, N_SOURCE)
        ids = pd.Index(np.concatenate([rankings[b][:a[b]] for b in range(nbins)])).sort_values()
        require(ids.is_unique and len(ids) == N_SOURCE and not set(ids)&set(target_ids), "Invalid source sample")
        sid = f"r{rep+1:02d}_{name}"
        source_sets.append({"sid":sid, "repeat":rep+1, "scenario":name, "tilt":tilt, "ids":ids})
        pd.DataFrame({"#IID":ids}).to_csv(OUT/"design"/f"{sid}.keep.tsv", sep="\t", index=False)
        observed = age["BMI"].loc[ids].to_numpy()
        bindex = np.searchsorted(edges, observed, side="right")-1
        realized_q = a/N_SOURCE
        diagnostic_weights = pi[bindex]/realized_q[bindex]
        ess = diagnostic_weights.sum()**2/(diagnostic_weights@diagnostic_weights)
        uncovered_mass = float(pi[(pi>0)&(a==0)].sum())
        sample_rows.append({"sid":sid, "repeat":rep+1, "scenario":name, "tilt":tilt, "N_source":len(ids),
                            "age_mean":observed.mean(), "age_SD":observed.std(ddof=1),
                            "target_age_mean":target_age.mean(), "age_mean_gap":observed.mean()-target_age.mean(),
                            "Wasserstein_years":stats.wasserstein_distance(observed,target_age),
                            "KS_statistic":stats.ks_2samp(observed,target_age).statistic,
                            "target_probability_in_unsampled_bins":uncovered_mass,
                            "age_bin_reweight_ESS_on_observed_support_only":ess})
        for b in range(nbins):
            quota_rows.append({"sid":sid, "bin_lower":edges[b], "bin_upper":edges[b+1],
                               "source_pool_count":pool_count[b], "target_count":target_count[b],
                               "target_probability":pi[b], "intended_source_probability":q[b], "N_sampled":a[b]})
sample_qc = pd.DataFrame(sample_rows)
age_quotas = pd.DataFrame(quota_rows)
age_quotas["realized_source_probability"] = age_quotas.N_sampled/N_SOURCE
age_quotas.to_csv(OUT/"design/age_bin_quotas.tsv",sep="\t",index=False)
age_proportions = age_quotas.loc[age_quotas.sid.str.startswith("r01_")].pivot(
    index=["bin_lower","bin_upper","target_probability"],columns="sid",values="realized_source_probability")
age_proportions.to_csv(OUT/"design/age_bin_proportions_wide.tsv",sep="\t")
print("Actual age-bin proportions (same quotas across repeats):")
display(age_proportions)
sample_qc.to_csv(OUT/"design/age_mixture_QC.tsv", sep="\t", index=False)
S = len(source_sets)
overlap = np.array([[len(set(a["ids"])&set(b["ids"])) for b in source_sets] for a in source_sets])
pd.DataFrame(overlap,index=[s["sid"] for s in source_sets],columns=[s["sid"] for s in source_sets]).to_csv(OUT/"design/source_overlap_counts.tsv",sep="\t")
design = {"seed":SEED, "N_target":len(target_ids), "N_source_each":N_SOURCE,
          "N_source_pool":len(source_pool), "repeats":N_REPEATS, "scenarios":SCENARIOS, "age_bin_years":AGE_BIN_YEARS,
          "source_P_threshold":IV_P_THRESHOLD, "reclumping":False,
          "N_boot":N_BOOT, "contrast_family":CONTRAST_FAMILY, "unrelated_keep":UNRELATED_KEEP,
          "nuisance":"saved full age spline + sex + QPC span", "target_age_mean":target_age.mean()}
write_json(OUT/"design/design.json", design)
display(pd.DataFrame([{k:design[k] for k in ("N_target","N_source_each","N_source_pool","repeats")}]))
display(sample_qc)

## 2. 每个 source 做 pooled BMI GWAS，按自己的 P 值选工具

PLINK 普通 OLS，保留完整 nuisance 空间、原始 phenotype 单位。`P < IV_P_THRESHOLD` 是唯一的工具相关性筛选；不重新 clump，不添加 F/MAF/MAC 筛选。每个 source 保存整个面板的 GWAS、所选工具清单、工具数量和工具集合重合度。无效回归明确记录；某个 source 没有工具时 MR unavailable，其他 source 继续。

In [ ]:
def full_rank_covariates(D):
    _, rankD, _ = nuisance_basis(D)
    kept=[]; X=np.ones((len(D),1)); rank=1
    for col in D:
        v=D[col].to_numpy()
        if np.std(v)<=1e-14: continue
        trial=np.column_stack([X,(v-v.mean())/v.std(ddof=1)])
        if np.linalg.matrix_rank(trial)>rank:
            kept.append(col); X=trial; rank+=1
    require(rank==rankD,"Nuisance span changed during rank reduction")
    Dc=D[kept].copy()
    return (Dc-Dc.mean())/Dc.std(ddof=1),kept,rankD

def select_source_instruments(x,threshold):
    valid=(x.ERRCODE.eq(".") & np.isfinite(x.P) & x.P.between(0,1)
           & np.isfinite(x.BETA) & np.isfinite(x.SE) & x.SE.gt(0)
           & (x.A1.eq(x.REF)|x.A1.eq(x.ALT)))
    return valid,valid & x.P.lt(threshold)

source_gwas,selection_rows={},[]
panel_lookup=panel.set_index("ID",drop=False)
for s in source_sets:
    sid,ids=s["sid"],s["ids"]
    folder=OUT/"source_GWAS"/sid; folder.mkdir(parents=True,exist_ok=True)
    pp,cp=folder/"BMI.pheno.tsv",folder/"BMI.covar.tsv"
    pheno["BMI"].loc[ids].to_csv(pp,sep="\t",index_label="#IID",header=True,float_format="%.17g")
    Dc,kept,rankD=full_rank_covariates(covar["BMI"].loc[ids])
    Dc.to_csv(cp,sep="\t",index_label="#IID",float_format="%.17g")
    write_json(folder/"input_signature.json",{"pheno_sha256":sha(pp),"covar_sha256":sha(cp),
        "keep_sha256":sha(OUT/"design"/f"{sid}.keep.tsv"),"kept_covariates":kept,
        "nuisance_rank":rankD,"P_threshold":IV_P_THRESHOLD,"reclumping":False})
    parts=[]
    for chrom in sorted(pfiles):
        chrom_panel=panel.loc[panel["#CHROM"].astype(int).eq(chrom)]
        if chrom_panel.empty: continue
        print(f"Source GWAS: {sid}, chr{chrom}",flush=True)
        prefix=folder/f"chr{chrom}"; extract=folder/f"chr{chrom}.panel.txt"
        chrom_panel.ID.to_csv(extract,index=False,header=False)
        run_cmd([PLINK,"--pfile",pfiles[chrom],"--keep",OUT/"design"/f"{sid}.keep.tsv",
            "--extract",extract,"--pheno",pp,"--pheno-name","BMI","--covar",cp,
            "--no-input-missing-phenotype","--glm","hide-covar",
            "--threads",THREADS,"--memory",MEMORY_MB,"--out",prefix],prefix)
        x=pd.read_csv(str(prefix)+".BMI.glm.linear",sep="\t",dtype=str,keep_default_na=False)
        require(x.TEST.eq("ADD").all() and x.ID.is_unique,"Expected one ADD row per SNP")
        for col in ("P","BETA","SE","OBS_CT"): x[col]=pd.to_numeric(x[col],errors="coerce")
        x["#CHROM"]=x["#CHROM"].str.replace(r"^chr","",regex=True)
        x["POS"]=pd.to_numeric(x.POS,errors="raise").astype(int)
        require(set(x.ID)==set(chrom_panel.ID),"GWAS panel coverage differs")
        for col in KEY:
            require(np.array_equal(x[col].astype(str).to_numpy(),panel_lookup.loc[x.ID,col].astype(str).to_numpy()),f"GWAS panel mismatch: {col}")
        x["F_classic"]=(x.BETA/x.SE)**2
        x["beta_ALT"]=x.BETA*np.where(x.A1.eq(x.ALT),1.,-1.)
        x["valid_GWAS"],x["selected_source_P"]=select_source_instruments(x,IV_P_THRESHOLD)
        parts.append(x)
    x=pd.concat(parts,ignore_index=True)
    require(x.ID.is_unique and set(x.ID)==set(panel.ID),"Source panel incomplete or duplicated")
    x.to_csv(folder/"pooled_BMI_GWAS.tsv",sep="\t",index=False)
    chosen=x.loc[x.selected_source_P].copy()
    chosen["BMI_increasing_allele"]=np.where(chosen.beta_ALT.gt(0),chosen.ALT,chosen.REF)
    chosen.to_csv(folder/"selected_IVs.tsv",sep="\t",index=False)
    source_gwas[sid]=x.set_index("ID",drop=False)
    s["iv_ids"]=pd.Index(chosen.ID)
    selection_rows.append({"sid":sid,"repeat":s["repeat"],"scenario":s["scenario"],
        "N_panel":len(x),"N_valid_GWAS":int(x.valid_GWAS.sum()),"N_selected":len(chosen),
        "P_threshold":IV_P_THRESHOLD,"min_F_classic":chosen.F_classic.min(),
        "mean_F_classic":chosen.F_classic.mean(),"N_selected_F_lt10":int(chosen.F_classic.lt(10).sum())})
selection=pd.DataFrame(selection_rows)
selection.to_csv(OUT/"qc/source_IV_selection.tsv",sep="\t",index=False)
display(selection)
union_ids=set().union(*(set(s["iv_ids"]) for s in source_sets))
iv=panel.loc[panel.ID.isin(union_ids),KEY].copy().sort_values(
    ["#CHROM","POS","ID"],key=lambda c:c.astype(int) if c.name in ("#CHROM","POS") else c).reset_index(drop=True)
K=len(iv)
iv.to_csv(OUT/"source_GWAS/union_selected_IVs.tsv",sep="\t",index=False)
require(K<=MAX_UNION_IV,"Union exceeds covariance memory guard; inspect counts/increase guard, never trim selected SNPs")
selection_overlap=[]
for a in source_sets:
    for b in source_sets:
        ia,ib=set(a["iv_ids"]),set(b["iv_ids"])
        selection_overlap.append({"source_a":a["sid"],"source_b":b["sid"],"K_a":len(ia),"K_b":len(ib),
            "intersection":len(ia&ib),"union":len(ia|ib),"Jaccard":len(ia&ib)/len(ia|ib) if ia|ib else np.nan})
pd.DataFrame(selection_overlap).to_csv(OUT/"qc/IV_set_overlap.tsv",sep="\t",index=False)
if K==0: warnings.warn("No significant source IV; MR unavailable. Threshold is not relaxed.")
print("Union of source-selected SNPs:",K)

## 3. 固定 target 关联与各 source 的等位基因对齐

只导出已选工具并集，明确计数 ALT dosage。在每个 source 复算已选 SNP 的 beta、普通 SE、N 并与 PLINK 核对；主 MR 使用原 GWAS beta/SE，按各自 source 的 BMI 升高等位基因定向。target 对并集只估计一次，不按 target P 筛选；不可估计的位点保存原因并排除。

HC1 influence 保留 source 之间重叠、共享 target 和 SNP 相关性，用于附加条件敏感性分析。需要个体独立，不能自动校正亲缘关系。

In [ ]:
# Explicit ALT-count export; default PLINK A export can count REF instead.
analysis_ids = pd.Index(sorted(set(target_ids).union(*(set(s["ids"]) for s in source_sets))))
pd.DataFrame({"#IID":analysis_ids}).to_csv(OUT/"design/genotype_export.keep.tsv",sep="\t",index=False)
G = pd.DataFrame(index=analysis_ids)
for chrom_s, part in iv.groupby("#CHROM",sort=False):
    chrom = int(chrom_s); prefix = OUT/"genotypes"/f"IV_chr{chrom}"
    extract = Path(str(prefix)+".extract.txt"); alleles = Path(str(prefix)+".ALT.tsv")
    part.ID.to_csv(extract,index=False,header=False)
    part[["ID","ALT"]].to_csv(alleles,sep="\t",index=False,header=False)
    run_cmd([PLINK,"--pfile",pfiles[chrom],"--keep",OUT/"design/genotype_export.keep.tsv",
             "--extract",extract,"--export","A","--export-allele",alleles,
             "--threads",THREADS,"--memory",MEMORY_MB,"--out",prefix],prefix)
    raw = pd.read_csv(str(prefix)+".raw",sep=r"\s+",dtype={"IID":str}).set_index("IID")
    require(raw.index.is_unique and set(raw.index)==set(analysis_ids),"Exported genotype IDs differ")
    if "FID" in raw: require(raw.FID.astype(str).eq("0").all(),"Unexpected genotype FID")
    for row in part.itertuples(index=False):
        col = f"{row.ID}_{row.ALT}"
        require(col in raw,f"ALT export allele unverified: {row.ID}")
        values = pd.to_numeric(raw.loc[analysis_ids,col],errors="raise")
        require(values.dropna().between(0,2).all(),"Invalid autosomal dosage")
        G[row.ID] = values
    del raw
G = G.loc[:,iv.ID]

def associations(y, C, dosage):
    """Marginal SNP regression adjusted for the full nuisance span; no Gxage.

    HC1 influence columns permit covariance across SNPs and overlapping samples.
    Each SNP uses its observed genotypes without mean imputation for regression.
    """
    y = np.asarray(y,float); C = np.asarray(C,float); A = np.asarray(dosage,float)
    n, k = A.shape
    IF = np.zeros((n,k)); rows = []; cache = {}
    for j in range(k):
        ok = np.isfinite(A[:,j]); nobs = int(ok.sum())
        token = np.packbits(ok).tobytes()
        if token not in cache:
            Q, rC, _ = nuisance_basis(C[ok])
            yr = y[ok]-Q@(Q.T@y[ok])
            cache[token] = (Q,rC,yr)
        Q, rC, yr = cache[token]
        g = A[ok,j]; gr = g-Q@(Q.T@g); den = float(gr@gr)
        df = nobs-rC-1
        row = {"N":nobs,"DF_RESID":df,"nuisance_rank":rC,"MAC":min(np.sum(g),2*nobs-np.sum(g))}
        if df<=5 or den<=1e-10 or nobs<MIN_ROLE_N:
            rows.append({**row,"status":"unidentifiable_or_too_small","BETA":np.nan,"SE_HC1":np.nan}); continue
        b = float(gr@yr/den); e = yr-b*gr
        influence = np.sqrt(nobs/df)*gr*e/den
        IF[ok,j] = influence
        se = float(np.linalg.norm(influence)); seclassic = float(np.sqrt((e@e/df)/den))
        rows.append({**row,"status":"ok" if se>0 and np.isfinite(se) else "invalid_SE", "BETA":b,"SE_HC1":se,
                     "SE_classic":seclassic,"P_HC1":2*stats.norm.sf(abs(b/se)) if se>0 else np.nan,
                     "P_classic":2*stats.t.sf(abs(b/seclassic),df) if seclassic>0 else np.nan,
                     "F_HC1":(b/se)**2 if se>0 else np.nan})
    cols=["N","DF_RESID","nuisance_rank","MAC","status","BETA","SE_HC1","SE_classic","P_HC1","P_classic","F_HC1"]
    return pd.DataFrame(rows).reindex(columns=cols), IF


target_assoc,target_cov={},{}
for outcome in OUTCOMES:
    a,influence=associations(pheno[outcome].loc[target_ids],covar[outcome].loc[target_ids],G.loc[target_ids])
    a=pd.concat([iv[KEY],a],axis=1)
    a["BETA_ALT"]=a.BETA
    a.to_csv(OUT/"associations"/f"fixed_target_{outcome}_ALT.tsv",sep="\t",index=False)
    target_assoc[outcome]=a
    target_cov[outcome]=influence.T@influence
    np.save(OUT/"associations"/f"fixed_target_{outcome}_HC1_cov.npy",target_cov[outcome])
    del influence
source_assoc,sparse_IF,strength_rows,audit_rows=[],[],[],[]
for s in source_sets:
    sid,ids,chosen=s["sid"],s["ids"],s["iv_ids"]
    a,influence=associations(pheno["BMI"].loc[ids],covar["BMI"].loc[ids],G.loc[ids,chosen])
    a=pd.concat([panel_lookup.loc[chosen,KEY].reset_index(drop=True),a],axis=1)
    require(a.status.eq("ok").all(),f"{sid}: selected source SNP refit failed")
    expected=source_gwas[sid].loc[chosen]
    beta_ok=np.allclose(a.BETA,expected.beta_ALT,rtol=5e-4,atol=5e-6)
    se_ok=np.allclose(a.SE_classic,expected.SE,rtol=5e-4,atol=5e-6)
    n_ok=np.array_equal(a.N.to_numpy(),expected.OBS_CT.to_numpy())
    audit_rows.append({"sid":sid,"K":len(a),"beta_agrees":beta_ok,"SE_agrees":se_ok,"N_agrees":n_ok})
    pd.DataFrame(audit_rows).to_csv(OUT/"qc/source_GWAS_refit_agreement.tsv",sep="\t",index=False)
    require(beta_ok and se_ok and n_ok,f"{sid}: GWAS/refit disagreement, inspect before MR")
    a["BETA_ALT"]=a.BETA
    a["orientation"]=np.where(expected.beta_ALT.to_numpy()>0,1.,-1.)
    a["BETA"]=expected.beta_ALT.to_numpy()*a.orientation
    a["SE_classic"]=expected.SE.to_numpy()
    a["P_GWAS"]=expected.P.to_numpy()
    a["sid"]=sid
    influence*=a.orientation.to_numpy()[None,:]
    a.to_csv(OUT/"associations"/f"{sid}_selected_BMI.tsv",sep="\t",index=False)
    source_assoc.append(a)
    ii=source_pool.get_indexer(ids); k=len(a)
    require((ii>=0).all(),"Source ID absent from pool")
    sparse_IF.append(sparse.csr_matrix((influence.ravel(),(np.repeat(ii,k),np.tile(np.arange(k),len(ii)))),shape=(len(source_pool),k)))
    strength_rows.append({"sid":sid,"K_selected":k,"mean_F_HC1":a.F_HC1.mean(),"min_F_HC1":a.F_HC1.min(),
        "N_F_HC1_lt10":int(a.F_HC1.lt(10).sum()),"min_MAC":a.MAC.min()})
    del influence
source_strength=pd.DataFrame(strength_rows)
source_strength.to_csv(OUT/"qc/source_IV_strength.tsv",sep="\t",index=False)
offsets=np.cumsum([0]+[len(a) for a in source_assoc])
IF_all=sparse.hstack(sparse_IF,format="csr")
SigmaX=(IF_all.T@IF_all).toarray()
require(np.allclose(np.sqrt(np.diag(SigmaX)),np.concatenate([a.SE_HC1.to_numpy() for a in source_assoc]),rtol=1e-9,atol=1e-12),"Source covariance diagonal mismatch")
np.save(OUT/"associations/source_joint_HC1_cov.npy",SigmaX)
write_json(OUT/"associations/source_covariance_order.json",{"sources":[
    {"sid":s["sid"],"IV_order":a.ID.tolist(),"start":int(offsets[j]),"end":int(offsets[j+1])}
    for j,(s,a) in enumerate(zip(source_sets,source_assoc))]})
del sparse_IF,IF_all,G
gc.collect()
display(source_strength)

## 4. 常规 MR 与条件敏感性

`MR_all_sources.tsv` 报告逐 source 常规 MR。IVW 使用普通 outcome SE 的 first-order 权重、multiplicative random-effects SE（单 SNP 为 Wald ratio）；Egger 使用各 source 的 BMI 升高方向与 MRE SE。conventional P 为大样本正态近似。少于 3 SNP 时本实现不给 median/Egger；无工具时均 unavailable。Median 点估计的区间在条件敏感性列中。

dIVW 和 IVW 使用同一套对齐后的工具。dIVW 将 IVW 分母中的 βX² 换成 βX² − SE_X²，使用原 BMI GWAS 的普通 SE_X。分母为非正或接近零时，该估计或重抽结果记为不可用。逐 source 表同时给出 [Ye, Shao & Kang (2021)](https://doi.org/10.1214/20-AOS2027) 式 (14) 的参考 SE。原文筛选版的理论依赖独立筛选数据。本分析用同一 BMI GWAS 筛选和估计 βX，因此 dIVW 作为弱工具误差校正的对照，winner's curse 仍然存在。

附加 Gaussian 系数重抽传播 βX/βY 误差并保留 source 重叠、共享 target；报告重复平均和相对 matched 的配对差异。**重抽固定观测到的各 source 工具集合，不重新 GWAS/选 SNP；区间和 P 不包含选择不确定性，不是完整流程 bootstrap 或选择后有效推断。** first-order NOME SE 也不修复 winner's curse。

32 个平均差异作 Bonferroni，文件名/列名带 conditional；校正不消除选择偏差。对整套选择流程作完整重抽需要每次重新 GWAS 和选 SNP，本版先复现常规流程的点估计。任何计划重复不可用时，不静默只平均其余重复。

In [ ]:
def weighted_median(x,y,se_y):
    x,y,se_y = np.asarray(x,float),np.asarray(y,float),np.asarray(se_y,float)
    w = x*x/(se_y*se_y)
    ok = np.isfinite(w)&(w>0)&np.isfinite(x)&np.isfinite(y)
    if not ok.any(): return np.nan
    ratio = y[ok]/x[ok]; w = w[ok]
    order = np.argsort(ratio,kind="stable"); ratio = ratio[order]; w = w[order]
    positions = (np.cumsum(w)-.5*w)/w.sum()
    return float(np.interp(.5,positions,ratio))

def mr_points(x,y,se_y,se_x):
    """Batch estimates; orientation fixed to each source observed BMI-increasing alleles."""
    X = np.atleast_2d(x); Y = np.atleast_2d(y)
    X,Y = np.broadcast_arrays(X,Y)
    if X.shape[1]==0: return {m:np.full(X.shape[0],np.nan) for m in METHODS}
    w = 1/np.asarray(se_y,float)**2
    numerator = np.sum(w*X*Y,axis=1)
    den = np.sum(w*X*X,axis=1)
    ivw = np.divide(numerator,den,out=np.full(len(X),np.nan),where=den>1e-20)
    # Ye, Shao & Kang (2021), equation (12), applied to the observed selected set.
    corrected_denominator = den - np.sum(w*np.asarray(se_x,float)**2)
    divw = np.divide(numerator,corrected_denominator,out=np.full(len(X),np.nan),
                     where=np.isfinite(corrected_denominator)&(corrected_denominator>1e-20))
    median = np.full(len(X),np.nan); egger = np.full(len(X),np.nan)
    if X.shape[1]>=3:
        median = np.array([weighted_median(a,b,se_y) for a,b in zip(X,Y)])
        sw = w.sum(); xm = np.sum(w*X,axis=1)/sw; ym = np.sum(w*Y,axis=1)/sw
        d = np.sum(w*(X-xm[:,None])**2,axis=1)
        egger = np.divide(np.sum(w*(X-xm[:,None])*(Y-ym[:,None]),axis=1),d,
                          out=np.full(len(X),np.nan),where=d>1e-20)
    return {"IVW":ivw,"dIVW":divw,"weighted_median":median,"MR_Egger":egger}

def mr_diagnostics(x,sx,y,sy):
    w = 1/sy**2; k = len(x); d = np.sum(w*x*x)
    theta = np.sum(w*x*y)/d
    Q = np.sum(w*(y-theta*x)**2)
    fese = 1/np.sqrt(d)
    rese = fese*np.sqrt(max(1,Q/(k-1))) if k>1 else fese
    out = {"IVW_Q":Q,"IVW_Q_df":k-1,"IVW_Q_P":stats.chi2.sf(Q,k-1) if k>1 else np.nan,
           "IVW_SE_FE_NOME":fese,"IVW_SE_MRE_NOME":rese,"max_IVW_weight_fraction":np.max(w*x*x)/d,
           "IVW_classic_CI_low":theta-1.959964*rese,"IVW_classic_CI_high":theta+1.959964*rese}
    if k>1:
        wx = 1/sx**2; bx = np.sum(wx*x)/wx.sum(); qx = np.sum(wx*(x-bx)**2)
        out["I2_GX"] = max(0,1-(k-1)/qx) if qx>0 else 0.
    else: out["I2_GX"] = np.nan
    if k>=3:
        sw = w.sum(); xm = np.sum(w*x)/sw; ym = np.sum(w*y)/sw
        dd = np.sum(w*(x-xm)**2)
        if dd>1e-20:
            slope = np.sum(w*(x-xm)*(y-ym))/dd; intercept = ym-slope*xm
            qe = np.sum(w*(y-intercept-slope*x)**2); scale = max(1,qe/(k-2))
            sei = np.sqrt(scale*(1/sw+xm*xm/dd)); ses = np.sqrt(scale/dd)
            out.update({"Egger_intercept":intercept,"Egger_intercept_SE_MRE_NOME":sei,
                        "Egger_intercept_P_normal":2*stats.norm.sf(abs(intercept/sei)),"Egger_SE_MRE_NOME":ses,
                        "Egger_Q":qe,"Egger_Q_df":k-2,"Egger_Q_P":stats.chi2.sf(qe,k-2)})
    return out

def covariance_root(V):
    if V.size==0: return V.copy()
    V = (V+V.T)/2
    lam,U = linalg.eigh(V,check_finite=False)
    require(lam.min()>=-1e-7*max(lam.max(),1e-20),"Covariance has materially negative eigenvalues")
    return U*np.sqrt(np.maximum(lam,0))[None,:]

def uncertainty(point,draws):
    good = draws[np.isfinite(draws)]
    if not np.isfinite(point) or len(good)<.99*len(draws):
        return {"status":"unavailable_or_unstable","SE_joint":np.nan,"CI_low":np.nan,"CI_high":np.nan,"P_joint_normal":np.nan,"N_boot_valid":len(good)}
    se = np.std(good,ddof=1)
    return {"status":"ok","SE_joint":se,"CI_low":point-1.959964*se,"CI_high":point+1.959964*se,
            "P_joint_normal":2*stats.norm.sf(abs(point/se)) if se>0 else np.nan,
            "bootstrap_percentile_low":np.quantile(good,.025),"bootstrap_percentile_high":np.quantile(good,.975),
            "bootstrap_median_minus_point":np.median(good)-point,"N_boot_valid":len(good)}


def conventional_mr(x,sx,y,sy):
    k=len(x)
    points={m:float(v[0]) for m,v in mr_points(x,y,sy,sx).items()}
    diag=mr_diagnostics(x,sx,y,sy) if k else {}
    divw_se = np.nan
    if k:
        w_hat = (x/sy)**2
        v_hat = (sx/sy)**2
        corrected_denominator = np.sum(w_hat-v_hat)
        diag["dIVW_denominator"] = corrected_denominator
        if np.isfinite(points["dIVW"]):
            # Reference variance, equation (14). Same-source selection is not covered.
            variance_numerator = np.sum(w_hat + points["dIVW"]**2*v_hat*(w_hat+v_hat))
            divw_se = np.sqrt(variance_numerator)/corrected_denominator

    standard_errors = {"IVW":diag.get("IVW_SE_MRE_NOME",np.nan),
                       "dIVW":divw_se,
                       "MR_Egger":diag.get("Egger_SE_MRE_NOME",np.nan)}
    inference = {"IVW":"first-order MRE/NOME", "MR_Egger":"first-order MRE/NOME",
                 "dIVW":"equation (14) reference SE with same-source screening",
                 "weighted_median":"see conditional sensitivity columns"}
    rows={}
    for method,point in points.items():
        se = standard_errors.get(method,np.nan)
        status = "ok" if np.isfinite(point) else "no_or_too_few_IVs"
        if method=="dIVW" and k and not np.isfinite(point):
            status = "unstable_dIVW_denominator"
        rows[method]={"estimate":point,"status":status,
            "SE_conventional":se,"CI_low_conventional":point-1.959964*se,"CI_high_conventional":point+1.959964*se,
            "P_conventional_normal":2*stats.norm.sf(abs(point/se)) if se>0 else np.nan,
            "inference":inference[method]}
    return rows,diag

# Conditional sensitivity only: observed selections/orientations are retained in draws.
Xhat=np.concatenate([a.BETA.to_numpy() for a in source_assoc])
rng=stable_rng("source-selected-joint-coefficients")
drawX=Xhat[None,:]+rng.standard_normal((N_BOOT,len(Xhat)))@covariance_root(SigmaX).T
mr_rows,mean_rows,contrast_rows,harmonization_rows=[],[],[],[]
for outcome in OUTCOMES:
    target=target_assoc[outcome]
    validT=target.status.eq("ok") & np.isfinite(target.BETA) & target.SE_classic.gt(0)
    target_mean=np.where(validT,target.BETA,0.)
    drawY=target_mean[None,:]+rng.standard_normal((N_BOOT,K))@covariance_root(target_cov[outcome]).T
    boots={m:np.full((N_BOOT,S),np.nan) for m in METHODS}
    points={m:np.full(S,np.nan) for m in METHODS}
    used_counts=[]
    for j,(s,a) in enumerate(zip(source_sets,source_assoc)):
        idx=pd.Index(iv.ID).get_indexer(a.ID)
        require((idx>=0).all(),"Source IV absent from union")
        ok=validT.to_numpy()[idx]; ia=np.flatnonzero(ok); iy=idx[ok]
        orient=a.orientation.to_numpy()[ok]
        x=a.BETA.to_numpy()[ok]; sx=a.SE_classic.to_numpy()[ok]
        y=target.BETA.to_numpy()[iy]*orient; sy=target.SE_classic.to_numpy()[iy]
        b=mr_points(drawX[:,offsets[j]:offsets[j+1]][:,ia],drawY[:,iy]*orient,sy,sx)
        conventional,diag=conventional_mr(x,sx,y,sy)
        k=len(x); used_counts.append(k)
        harmonization_rows.append({"outcome":outcome,"sid":s["sid"],"K_selected":len(a),"K_used":k,"N_invalid_target":int((~ok).sum())})
        aligned=a.copy(); aligned["target_valid"]=ok
        aligned["target_BETA_ALT"]=target.BETA.to_numpy()[idx]
        aligned["target_SE_classic"]=target.SE_classic.to_numpy()[idx]
        aligned.to_csv(OUT/"associations"/f"{s['sid']}_{outcome}_harmonized.tsv",sep="\t",index=False)
        for method in METHODS:
            point=conventional[method]["estimate"]
            boots[method][:,j]=b[method]; points[method][j]=point
            sensitivity={"conditional_"+key:value for key,value in uncertainty(point,b[method]).items()}
            mr_rows.append({"outcome":outcome,"sid":s["sid"],"repeat":s["repeat"],"scenario":s["scenario"],
                "method":method,"K_selected":len(a),"K_used":k,**conventional[method],**sensitivity,**diag})
    np.savez_compressed(OUT/"MR"/f"{outcome}_conditional_MR_draws.npz",**boots)
    for name,_ in SCENARIOS:
        jj=[j for j,s in enumerate(source_sets) if s["scenario"]==name]
        for method in METHODS:
            p=points[method][jj]; point=float(np.mean(p)); bd=np.mean(boots[method][:,jj],axis=1)
            mean_rows.append({"outcome":outcome,"scenario":name,"method":method,"estimate_mean_repeats":point,
                "K_used_min":int(min(used_counts[j] for j in jj)),"K_used_max":int(max(used_counts[j] for j in jj)),
                "N_repeats_available":int(np.isfinite(p).sum()),"inference_scope":"conditional on observed source-specific selection",
                **uncertainty(point,bd)})
    for name,_ in SCENARIOS:
        if name=="matched": continue
        for method in METHODS:
            ds,db=[],[]
            for rep in range(1,N_REPEATS+1):
                j=next(j for j,s in enumerate(source_sets) if s["repeat"]==rep and s["scenario"]==name)
                m=next(j for j,s in enumerate(source_sets) if s["repeat"]==rep and s["scenario"]=="matched")
                delta=points[method][j]-points[method][m]; bd=boots[method][:,j]-boots[method][:,m]
                contrast_rows.append({"outcome":outcome,"scenario":name,"method":method,"repeat":rep,"delta":delta,
                    "inference_scope":"conditional on observed source-specific selection",**uncertainty(delta,bd)})
                ds.append(delta); db.append(bd)
            delta=float(np.mean(ds)); bd=np.mean(db,axis=0)
            contrast_rows.append({"outcome":outcome,"scenario":name,"method":method,"repeat":"mean","delta":delta,
                "inference_scope":"conditional on observed source-specific selection",**uncertainty(delta,bd)})
mr=pd.DataFrame(mr_rows).merge(sample_qc,on=["sid","repeat","scenario"],validate="many_to_one")
mean_mr=pd.DataFrame(mean_rows); contrasts=pd.DataFrame(contrast_rows)
contrasts["P_Bonf_planned_contrasts_conditional"]=np.where(contrasts.repeat.eq("mean"),np.minimum(contrasts.P_joint_normal*CONTRAST_FAMILY,1),np.nan)
pd.DataFrame(harmonization_rows).to_csv(OUT/"qc/target_harmonization.tsv",sep="\t",index=False)
mr.to_csv(OUT/"MR/MR_all_sources.tsv",sep="\t",index=False)
mean_mr.to_csv(OUT/"MR/MR_mean_repeats_conditional.tsv",sep="\t",index=False)
contrasts.to_csv(OUT/"MR/paired_age_mismatch_contrasts_conditional.tsv",sep="\t",index=False)
display(mr[["outcome","sid","method","K_selected","K_used","estimate","SE_conventional","P_conventional_normal"]])
display(mean_mr)
display(contrasts.loc[contrasts.repeat.eq("mean")])

## 5. 输出与解释

- `design/age_bin_proportions_wide.tsv`：target 和五种 source 的实际年龄箱比例。
- `qc/source_IV_selection.tsv`、`qc/IV_set_overlap.tsv`：工具数、F、集合重合度。
- `source_GWAS/{sid}/pooled_BMI_GWAS.tsv`、`selected_IVs.tsv`：各 source 的完整 GWAS 与工具。
- `MR/MR_all_sources.tsv`：逐 source 常规 MR，以及附加 conditional 敏感性区间。
- `MR/MR_mean_repeats_conditional.tsv`、`paired_age_mismatch_contrasts_conditional.tsv`：重复平均和配对差异。

由于 discovery 池并回，实际 source N 可能比 v2 大，跨版本变化不能只归因于工具选择规则。matched 不是因果真值；这里的比较同时包含年龄、样本构成、工具选择与强度变化。现有 LD pruning 不保证当前样本完全无 LD，本版按用户要求不重做 pruning，需保留原始面板来源。

本地验证范围见仓库 README；合成数据和旧版真实结果都不作为本版 AoU 结果。

In [ ]:
scenario_names = [name for name,_ in sorted(SCENARIOS,key=lambda z:z[1])]
labels = {"younger_strong":"Younger (strong)","younger_moderate":"Younger (moderate)",
          "matched":"Age matched","older_moderate":"Older (moderate)","older_strong":"Older (strong)"}
colors = {"IVW":"#245b9e","dIVW":"#c16b19","weighted_median":"#14866d","MR_Egger":"#a24582"}

fig,ax = plt.subplots(figsize=(8,4))
for name in scenario_names:
    a = sample_qc.loc[sample_qc.scenario.eq(name)]
    ax.scatter(a.age_mean,a.age_SD,label=labels[name],s=45)
ax.axvline(target_age.mean(),color="black",ls="--",lw=1,label="Fixed BP target mean age")
ax.set(xlabel="BMI source mean age (years)",ylabel="BMI source age SD (years)",title=f"Age-mixture sampling; N = {N_SOURCE:,} per source")
ax.legend(fontsize=8,bbox_to_anchor=(1.01,1),loc="upper left")
fig.tight_layout(); fig.savefig(OUT/"figures/age_mixture_QC.png",dpi=180,bbox_inches="tight"); plt.show()


fig,ax=plt.subplots(figsize=(9,5))
bin_centers=np.array([(a+b)/2 for a,b,_ in age_proportions.index])
ax.plot(bin_centers,age_proportions.index.get_level_values("target_probability"),color="black",ls="--",label="BP target")
for name in scenario_names:
    ax.plot(bin_centers,age_proportions[f"r01_{name}"],marker=".",label=labels[name])
ax.set(xlabel="Age-bin midpoint (years)",ylabel="Proportion",title="Actual five-year age-bin proportions")
ax.legend(); fig.tight_layout(); fig.savefig(OUT/"figures/age_bin_proportions.png",dpi=180); plt.show()

for outcome in OUTCOMES:
    # Separate axes prevent a very imprecise Egger estimate hiding the IVW scale.
    fig,axes = plt.subplots(1,len(METHODS),figsize=(4.3*len(METHODS),4.3))
    for ax,method in zip(axes,METHODS):
        a = mean_mr.loc[mean_mr.outcome.eq(outcome)&mean_mr.method.eq(method)].set_index("scenario").reindex(scenario_names)
        good = a.status.eq("ok")
        yy = np.arange(len(a))
        ax.errorbar(a.loc[good,"estimate_mean_repeats"],yy[good],xerr=1.959964*a.loc[good,"SE_joint"],fmt="o",color=colors[method],capsize=3)
        ax.set_yticks(yy,[labels[name] for name in scenario_names] if ax is axes[0] else [""]*len(yy))
        ax.axvline(0,color="grey",lw=.7); ax.invert_yaxis()
        ax.set(title=method,xlabel="mmHg per BMI unit")
        if not good.all(): ax.text(.02,.02,"Some estimates unavailable",transform=ax.transAxes,fontsize=8)
    fig.suptitle(f"BMI → {outcome}: source-specific P-selected IVs; fixed target; mean over paired source repeats\n95% CIs conditional on observed selections; no re-selection in draws",fontsize=11)
    fig.tight_layout(); fig.savefig(OUT/"figures"/f"MR_{outcome}.png",dpi=180,bbox_inches="tight"); plt.show()
    fig,axes = plt.subplots(1,len(METHODS),figsize=(4.3*len(METHODS),4.3))
    names = [name for name in scenario_names if name!="matched"]
    for ax,method in zip(axes,METHODS):
        a = contrasts.loc[contrasts.outcome.eq(outcome)&contrasts.method.eq(method)&contrasts.repeat.eq("mean")].set_index("scenario").reindex(names)
        good = a.status.eq("ok"); yy = np.arange(len(a))
        ax.errorbar(a.loc[good,"delta"],yy[good],xerr=1.959964*a.loc[good,"SE_joint"],fmt="o",color=colors[method],capsize=3)
        ax.set_yticks(yy,[labels[name] for name in names] if ax is axes[0] else [""]*len(yy))
        ax.axvline(0,color="black",lw=1); ax.invert_yaxis()
        ax.set(title=method,xlabel="Difference from age-matched MR")
    fig.suptitle(f"{outcome}: paired age-mismatch contrasts\nShared target and source overlap included in joint resampling",fontsize=11)
    fig.tight_layout(); fig.savefig(OUT/"figures"/f"paired_contrasts_{outcome}.png",dpi=180,bbox_inches="tight"); plt.show()

write_json(OUT/"ANALYSIS_COMPLETE.json",{
    "status":"complete","finished_utc":datetime.now(timezone.utc).isoformat(),"design":design,
    "linear_run":str(LINEAR_RUN),"plink_version":PLINK_VERSION,"K_union":K,"N_MR_rows":len(mr), "selection_uncertainty_included":False,
    "N_mean_contrasts":int(contrasts.repeat.eq("mean").sum()),
    "inference":"First-order MRE/NOME, dIVW reference SE, and HC1 paired sensitivity conditional on observed source selections",
    "interpretation":"Age composition plus instrument-selection sensitivity; matched is not causal truth"})
print("B3 v3 completed:",OUT)
print("Main tables: MR/MR_all_sources.tsv; means/contrasts have _conditional suffix")